In [ ]:
import sys

sys.path.append("..")
from pyspark.sql import functions as F

from src.paths import BRONZE, RAW
from src.schemas import SCHEMAS
from src.spark import get_spark

spark = get_spark()
spark

26/09/16 09:53:00 WARN Utils: Your hostname, Leventes-MacBook-Air-M4.local resolves to a loopback address: 127.0.0.1; using 172.20.10.14 instead (on interface en0)
26/09/16 09:53:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 09:53:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
bronze = {}
for name, (csv, schema) in SCHEMAS.items():
    df = spark.read.csv(str(RAW / csv), header=True, schema=schema)
    df.write.mode("overwrite").parquet(str(BRONZE / name))
    bronze[name] = spark.read.parquet(str(BRONZE / name))
    print(f"{name:13s} {bronze[name].count():>9,d} rows")

sales         3,000,888 rows
stores               54 rows
oil               1,218 rows
holidays            350 rows
transactions     83,488 rows


In [3]:
bronze["sales"].printSchema()

root
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- family: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- onpromotion: integer (nullable = true)



In [4]:
sales = bronze["sales"]
sales.describe("sales").show()
zero_share = sales.filter(F.col("sales") == 0).count() / sales.count()
print(f"zero-sales share: {zero_share:.1%}")

+-------+-----------------+
|summary|            sales|
+-------+-----------------+
|  count|          3000888|
|   mean|357.7757491126189|
| stddev|1101.997721338003|
|    min|              0.0|
|    max|         124717.0|
+-------+-----------------+

zero-sales share: 31.3%


In [5]:
sales.groupBy("family").agg(F.sum("sales").alias("total")).orderBy(F.desc("total")).show(33, truncate=False)

+--------------------------+--------------------+
|family                    |total               |
+--------------------------+--------------------+
|GROCERY I                 |3.434627348859998E8 |
|BEVERAGES                 |2.16954486E8        |
|PRODUCE                   |1.2270468467645992E8|
|CLEANING                  |9.7521289E7         |
|DAIRY                     |6.4487709E7         |
|BREAD/BAKERY              |4.2133945576369E7   |
|POULTRY                   |3.187600447172101E7 |
|MEATS                     |3.1086468404074874E7|
|PERSONAL CARE             |2.4592051E7         |
|DELI                      |2.4110322468767E7   |
|HOME CARE                 |1.6022744E7         |
|EGGS                      |1.5588296E7         |
|FROZEN FOODS              |1.4073887719910512E7|
|PREPARED FOODS            |8799895.116942499   |
|LIQUOR,WINE,BEER          |7746640.0           |
|SEAFOOD                   |2015431.8828235997  |
|GROCERY II                |1962767.0           |


In [7]:
sales.filter((F.month("date") == 12) & (F.dayofmonth("date") == 25)).count()   # expect 0

0

In [8]:
sales.select("date").distinct().count()   # expect 1684 = 1688 days - 4 Christmases

1684

In [9]:
first_sale = (sales.filter(F.col("sales") > 0)
                   .groupBy("store_nbr").agg(F.min("date").alias("first_sale_date"))
                   .orderBy(F.desc("first_sale_date")))
first_sale.show(10)

+---------+---------------+
|store_nbr|first_sale_date|
+---------+---------------+
|       52|     2017-04-20|
|       22|     2015-10-09|
|       42|     2015-08-21|
|       21|     2015-07-24|
|       29|     2015-03-20|
|       20|     2015-02-13|
|       53|     2014-05-29|
|       36|     2013-05-09|
|       38|     2013-01-02|
|       12|     2013-01-02|
+---------+---------------+
only showing top 10 rows



In [11]:
oil = bronze["oil"]
print(oil.count(), oil.filter(F.col("dcoilwtico").isNull()).count())   # 1218 rows, 43 nulls

1218 43


In [12]:
oil.select(F.min("date"), F.max("date")).show()

+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2013-01-01|2017-08-31|
+----------+----------+

